# Kommentar-Datensatz vorbereiten

Dieses Notebook führt die KI- und Real-Kommentardateien zusammen, kennzeichnet die Gruppe, prueft die Verteilung und speichert einen gemeinsamen Datensatz für die weitere Analyse.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path


In [ ]:
DATA_DIR = Path('../data')
AI_PATH = DATA_DIR / 'tiktok-comments-dataset-ai.csv'
REAL_PATH = DATA_DIR / 'tiktok-comments-dataset-real.csv'
OUTPUT_PATH = DATA_DIR / 'tiktok-comments-full.csv'

DO_MATCH_VIDEOS = True
RANDOM_SEED = 42


In [ ]:
# Beide Kommentardateien laden und direkt mit einem Gruppelabel versehen.
df_ai = pd.read_csv(AI_PATH).copy()
df_real = pd.read_csv(REAL_PATH).copy()

df_ai['influencer_type'] = 'ai'
df_real['influencer_type'] = 'real'

print(f'AI comments loaded: {len(df_ai):,}')
print(f'Real comments loaded: {len(df_real):,}')


In [ ]:
df = pd.concat([df_ai, df_real], ignore_index=True)

print(f'Total comments in merged dataset: {len(df):,}')
print('Available groups:', sorted(df['influencer_type'].dropna().unique().tolist()))

In [ ]:
# Kurzer Strukturcheck
video_counts = df.groupby('influencer_type')['post_url'].nunique().rename('n_videos')
comment_counts = df.groupby('influencer_type').size().rename('n_comments')

summary_counts = pd.concat([video_counts, comment_counts], axis=1)
display(summary_counts)

comments_per_video = (
    df.groupby(['influencer_type', 'post_url'])
      .size()
      .reset_index(name='n_comments')
)

display(comments_per_video.groupby('influencer_type')['n_comments'].describe().round(2))


In [ ]:
# Video-Matching: Beide Gruppen auf gleich viele Videos begrenzen.
if DO_MATCH_VIDEOS:
    rng = np.random.default_rng(RANDOM_SEED)

    videos_ai = df.loc[df['influencer_type'] == 'ai', 'post_url'].dropna().unique()
    videos_real = df.loc[df['influencer_type'] == 'real', 'post_url'].dropna().unique()
    n_match = min(len(videos_ai), len(videos_real))

    sampled_ai = rng.choice(videos_ai, n_match, replace=False)
    sampled_real = rng.choice(videos_real, n_match, replace=False)
    matched_urls = np.concatenate([sampled_ai, sampled_real])

    df = df[df['post_url'].isin(matched_urls)].copy()

    print('After video matching:')
    display(pd.concat([
        df.groupby('influencer_type')['post_url'].nunique().rename('n_videos'),
        df.groupby('influencer_type').size().rename('n_comments'),
    ], axis=1))
else:
    print('Video matching skipped.')


In [ ]:
# Den vorbereiteten Gesamtdatensatz speichern.
df.to_csv(OUTPUT_PATH, index=False)
print(f'Saved merged comment dataset to: {OUTPUT_PATH.resolve()}')
